<a href="https://colab.research.google.com/github/DiyaRana7/Flyrank_ML/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use a Random Forest classification model for the Refresh / Content Opportunity Scoring lane.

The model will estimate whether a content page meets a defined future-performance condition that makes it worth prioritizing for human review.

Random Forest is appropriate for this task because the relationship between search visibility, engagement, content age, and future performance may not be purely linear. It can combine several observable signals and capture interactions between them without requiring a very complex model.

The model will be used for decision support. It is not intended to predict Google's ranking algorithm or prove that refreshing a page causes better performance.

I will compare the model with the transparent rule-based baseline created in Week 4. The model will only be considered useful if it performs better than the baseline on the same evaluation data and metric.

In [15]:
!git clone https://github.com/DiyaRana7/Flyrank_ML.git
%cd Flyrank_ML

Cloning into 'Flyrank_ML'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 124 (delta 37), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.86 MiB | 5.24 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/Flyrank_ML/Flyrank_ML


In [16]:
import pandas as pd
import numpy as np
import os

# Make sure we are inside the repository
REPO_PATH = "/content/Flyrank_ML"

if os.path.exists(REPO_PATH):
    os.chdir(REPO_PATH)
else:
    raise FileNotFoundError(
        "Flyrank_ML repository not found. Clone the repository first."
    )

# Load the starter dataset used for the baseline
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [17]:
candidate_features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc"
]

available_features = [
    col for col in candidate_features
    if col in df.columns
]

print("Available candidate features:")
print(available_features)

print("\nMissing candidate features:")
print([
    col for col in candidate_features
    if col not in df.columns
])

Available candidate features:
['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc']

Missing candidate features:
[]


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **time-aware split** so that the model is trained on earlier observations and evaluated on later observations.

This is more appropriate for a refresh-prioritization task because the model should only use information that would have been available before the decision.

I will keep the evaluation data separate from the training data and use the same evaluation period and metric when comparing the Random Forest with my Week-4 baseline.

I will not use future outcome fields such as `trend_direction` or `trend_pct` as model features.

In [18]:
from sklearn.model_selection import GroupShuffleSplit

# Target: whether the observed trend is declining
# This is the outcome we want the model to identify.
y = (df["trend_direction"] == "down").astype(int)

# Features available before the decision
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_cols].copy()

# Handle missing numeric values using medians
X = X.fillna(X.median(numeric_only=True))

# Group by client so the same client does not appear in both train and test
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.33,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining declining rate:", round(y_train.mean(), 3))
print("Test declining rate:", round(y_test.mean(), 3))

print("\nTraining clients:")
print(df.iloc[train_idx]["client_id"].nunique())

print("Test clients:")
print(df.iloc[test_idx]["client_id"].nunique())

Training rows: 18690
Test rows: 11310

Training declining rate: 0.546
Test declining rate: 0.536

Training clients:
21
Test clients:
11


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model

I will train a Random Forest classifier to identify pages whose observed trend is declining.

The target is `1` when `trend_direction == "down"` and `0` otherwise.

The model uses only pre-decision performance and content signals. `trend_direction` and `trend_pct` are excluded from the feature set because they contain outcome information.

I will compare the model with the Week-4 rule-based baseline on the same test rows.

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

# Model predictions
model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

Model trained successfully.


In [20]:
model_metrics = {
    "accuracy": accuracy_score(y_test, model_pred),
    "precision": precision_score(y_test, model_pred, zero_division=0),
    "recall": recall_score(y_test, model_pred, zero_division=0),
    "f1": f1_score(y_test, model_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, model_prob)
}

print("Random Forest results:")
for metric, value in model_metrics.items():
    print(f"{metric}: {value:.3f}")

Random Forest results:
accuracy: 0.878
precision: 0.839
recall: 0.957
f1: 0.894
roc_auc: 0.968


In [21]:
# Recreate the Week-4 baseline on exactly the same test rows

baseline_test = df.iloc[test_idx].copy()

baseline_test["baseline_score"] = 0

baseline_test["baseline_score"] += (
    baseline_test["days_since_last_update"] >= 180
).astype(int) * 2

baseline_test["baseline_score"] += (
    baseline_test["days_since_last_update"] >= 365
).astype(int)

baseline_test["baseline_score"] += (
    baseline_test["impressions_90d"] >= 500
).astype(int) * 2

baseline_test["baseline_score"] += (
    baseline_test["impressions_90d"] >= 5000
).astype(int)

baseline_test["baseline_score"] += (
    baseline_test["avg_position"] > 10
).astype(int)

# Week-4 action
baseline_test["baseline_action"] = (
    baseline_test["baseline_score"] >= 4
).astype(int)

baseline_pred = baseline_test["baseline_action"]

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0)
}

print("Week-4 baseline results:")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.3f}")

Week-4 baseline results:
accuracy: 0.479
precision: 0.549
recall: 0.157
f1: 0.245


In [22]:
comparison = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "f1"],
    "Week-4 baseline": [
        baseline_metrics["accuracy"],
        baseline_metrics["precision"],
        baseline_metrics["recall"],
        baseline_metrics["f1"]
    ],
    "Random Forest": [
        model_metrics["accuracy"],
        model_metrics["precision"],
        model_metrics["recall"],
        model_metrics["f1"]
    ]
})

display(comparison.round(3))

,metric,Week-4 baseline,Random Forest
0,accuracy,0.479,0.878
1,precision,0.549,0.839
2,recall,0.157,0.957
3,f1,0.245,0.894


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The model's predictions should be interpreted as decision-support rather than proof that a page needs a refresh.

I will inspect false positives and false negatives to understand where the model makes mistakes.

I will also inspect feature importance to understand which observable signals the model relies on most.

A feature being important in this model does not establish that it causes search performance changes.

In [23]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, model_pred)

print("Confusion matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nFalse positives:", fp)
print("False negatives:", fn)

Confusion matrix:
[[4133 1116]
 [ 261 5800]]

False positives: 1116
False negatives: 261


In [24]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Top model features:")
display(importance.head(10))

Top model features:


,feature,importance
10,impressions_prev_30d,0.236295
7,impressions_last_30d,0.195888
0,impressions_90d,0.087493
13,content_age_days,0.079001
16,avg_position,0.068677
5,days_with_impressions,0.061798
9,sessions_last_30d,0.034304
18,scroll_rate,0.030313
15,ctr,0.027462
2,sessions_90d,0.022537


In [25]:
error_df = df.iloc[test_idx].copy()

error_df["actual_declining"] = y_test.values
error_df["predicted_declining"] = model_pred

false_positives = error_df[
    (error_df["actual_declining"] == 0) &
    (error_df["predicted_declining"] == 1)
]

false_negatives = error_df[
    (error_df["actual_declining"] == 1) &
    (error_df["predicted_declining"] == 0)
]

print("Sample false positives:")
display(
    false_positives[
        [
            "content_id",
            "impressions_90d",
            "avg_position",
            "ctr",
            "days_since_last_update",
            "trend_direction"
        ]
    ].head(5)
)

print("Sample false negatives:")
display(
    false_negatives[
        [
            "content_id",
            "impressions_90d",
            "avg_position",
            "ctr",
            "days_since_last_update",
            "trend_direction"
        ]
    ].head(5)
)

Sample false positives:


,content_id,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction
21,content_9d548144b06d,86,12.6,0.00,20,stable
26,content_72c5c2d73e5a,2426,30.0,0.12,13,stable
34,content_55f75c034970,3998,6.4,0.03,8,up
36,content_bce275871a25,371,5.4,1.35,20,stable
110,content_0080113aa348,15176,19.1,0.94,104,up


Sample false negatives:


,content_id,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction
252,content_aba4b4460e47,1334,50.1,0.00,22,down
491,content_b65c621c6c0b,687,14.2,0.29,7,down
657,content_e662d215360f,1854,11.4,1.19,104,down
826,content_7c9b3e0182d1,27563,8.0,0.25,25,down
844,content_3d49ab508c58,720,41.5,0.00,104,down


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

all done